# Python Garbage Collection

## 1. What is garbage collection?

When objects are no longer needed and cannot be reached through active references, their memory can eventually be reclaimed.

**CPython** uses reference counting together with a cyclic garbage collector.

In [1]:
numbers = [1, 2, 3]
print(numbers)

numbers = None
print("The list is no longer referenced by numbers.")

[1, 2, 3]
The list is no longer referenced by numbers.


## 2. References

Variables are names that refer to objects. Two variables can refer to the same object.

In [2]:
a = [10, 20]
b = a

print(a is b)
b.append(30)

print(a)
print(b)

True
[10, 20, 30]
[10, 20, 30]


## 3. `del`

`del` removes a name/reference. It does not necessarily destroy the object if another reference still exists.

In [3]:
data = [1, 2, 3]
other = data

del data

print(other)

[1, 2, 3]


## 4. Reference counting

In CPython, reference counting is an important part of memory management. When an object's reference count reaches zero, its memory can be reclaimed.

`sys.getrefcount()` itself temporarily adds a reference, so its result is usually one higher than expected.

In [4]:
import sys

obj = []
print(sys.getrefcount(obj))

other = obj
print(sys.getrefcount(obj))

del other
print(sys.getrefcount(obj))

2
3
2


## 5. Reference cycles

Reference counting alone cannot handle a cycle where objects refer to one another. The cyclic garbage collector can detect unreachable cycles.

In [5]:
class Node:
    def __init__(self, name):
        self.name = name
        self.other = None

a = Node("A")
b = Node("B")
a.other = b
b.other = a

del a
del b

print("External references to the cycle are gone.")

External references to the cycle are gone.


## 6. The `gc` module

Useful functions:
- `gc.collect()` → manually request collection
- `gc.enable()` → enable automatic cyclic GC
- `gc.disable()` → disable automatic cyclic GC
- `gc.isenabled()` → check whether it is enabled
- `gc.get_count()` → inspect GC counters
- `gc.get_threshold()` → inspect thresholds

In [6]:
import gc

print("GC enabled:", gc.isenabled())
print("GC counters:", gc.get_count())
print("GC thresholds:", gc.get_threshold())
print("Collection result:", gc.collect())

GC enabled: True
GC counters: (1293, 3, 4)
GC thresholds: (2000, 10, 10)
Collection result: 340


## 7. Enable and disable GC

`gc.disable()` disables **automatic cyclic garbage collection**. It does not stop all Python memory management.

In [7]:
import gc

print("Before:", gc.isenabled())
gc.disable()
print("After disable:", gc.isenabled())
gc.enable()
print("After enable:", gc.isenabled())

Before: True
After disable: False
After enable: True


## 8. Checking tracked objects

`gc.is_tracked(obj)` tells you whether an object is currently tracked by the cyclic garbage collector.

In [8]:
import gc

values = [10, "hello", [1, 2, 3], {"a": 1}]

for value in values:
    print(type(value).__name__, ":", gc.is_tracked(value))

int : False
str : False
list : True
dict : True


## 9. GC thresholds

`gc.get_threshold()` shows the thresholds used by automatic collection. `gc.set_threshold()` can change them, but normally the defaults should be left alone unless measurement gives a reason to tune them.

In [9]:
import gc

old = gc.get_threshold()
print("Current:", old)

gc.set_threshold(*old)
print("Restored:", gc.get_threshold())

Current: (2000, 10, 10)
Restored: (2000, 10, 10)


## 10. Garbage collection vs resource cleanup

Garbage collection manages memory for unreachable objects. It is **not** a replacement for deterministic cleanup of resources.

Use context managers (`with`) for files, sockets, locks, database connections, etc.

In [10]:
from io import StringIO

with StringIO("Hello") as file:
    print(file.read())

print("The context manager handled cleanup.")

Hello
The context manager handled cleanup.
